# Partition reanalysis uncertainty between impact model training and GCM post-processing

Adapting code from 3_project_impacts.ipynb to allow cross-projection between different impact model training and bias-correcting/downscaling reanalysis sources - but only keeping the 'main' specification for simplicity

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar
from funcs_aux import filter_problems

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_readregs import coeffs_mortality, coeffs_gdppc, coeffs_payroll
dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39983 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/crhtc70.hpc.ucar.edu/40728/proxy/39983/status,
Dashboard: /node/crhtc70.hpc.ucar.edu/40728/proxy/39983/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:32791,Workers: 0
Dashboard: /node/crhtc70.hpc.ucar.edu/40728/proxy/39983/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35921,Total threads: 3
Dashboard: /node/crhtc70.hpc.ucar.edu/40728/proxy/34567/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:43555,


In [3]:
params_proc = {'mortality':{'varname':'tassumpoly','geo_typ':'impact-regions','popwght':True},
               'gdppc':{'varname':'tasmeanpoly','geo_typ':'adm1','popwght':True},
               'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}
#params_proc = {'payroll':{'varname':'days90plus','geo_typ':'adm2-us','popwght':True}}
#{'varname':'tasmaxbinF','geo_typ':'adm2-us','popwght':True}}

# Whether to nan out problem members of the BCD-ME
filter_bcd_me_flags = False # moot by fixed internal BCD-ME files
problem_members = xr.open_dataset(f'{dir_list['aux']}problem_member_flags_qdm-qplad.nc')

In [4]:
overwrite = True

for proc in params_proc:
    # Get filenames of aggregated climate predictors
    fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/{params_proc[proc]['varname']}_*_{params_proc[proc]['geo_typ']}*{'*popwght*'*params_proc[proc]['popwght']}.zarr'))
    if params_proc[proc]['popwght']:
        fns = [fn for fn in fns if re.search(r'\_popwght\.zarr',fn)]
    else:
        fns = [fn for fn in fns if not re.search(r'\_popwght\.zarr',fn)]
    fns = {re.split(r'\_',re.split(r'\/',fn)[-1])[2]:fn for fn in fns}
    
    # Load regression coefficients
    if proc == 'gdppc':
        coeffs,vcv = coeffs_gdppc(return_raw=False)
    elif proc == 'mortality':
        coeffs,vcv = coeffs_mortality(return_raw=True)
        # Keep the Ind flag to test something
        sub_params = {'resolution':'025deg','timeframe':'1980-2010','obstreat':'sep','spec':'pooled'}
        coeffs = coeffs.sel(sub_params)
        vcv = vcv.sel(sub_params)
    elif proc == 'payroll':
        coeffs,vcv = coeffs_payroll(return_raw=False)
    else:
        raise KeyError()
        
    # Rename proj_bases to allow for cross-processing
    coeffs = coeffs.rename({'proj_base':'fit_mod'})
    
    for mod in tqdm(fns):
        output_fn = f'{dir_list['proc']}impact_projs/{proc}/{re.sub(params_proc[proc]['varname'],f'd{proc}-crossfit',re.split(r'\/',fns[mod])[-1])}'
        
        if overwrite or (not os.path.exists(output_fn)):
            
            # Load aggregated climate predictor
            ds = xr.open_zarr(fns[mod],consolidated=False,decode_timedelta=False)

            # Set BCD-ME ensemble members to nan if they've been identified
            # as having remaining post-publishing errors
            if filter_bcd_me_flags:
                flag_var = ('tas' if proc in ['mortality','gdppc'] else 'tasmax')
                ds[params_proc[proc]['varname']] = filter_problems(ds[[params_proc[proc]['varname']]].reset_index('idv'),
                                                                   problem_members,
                                                                   verbose=True,flag_var=flag_var)[params_proc[proc]['varname']]
        
            # Isolate predictor variables
            if params_proc[proc]['varname'] == 'tassumpoly':
                ds['degree'] = [f'tassumpoly_{k}' for k in ds.degree.values]
                ds = ds.rename({'degree':'pred'})
            elif params_proc[proc]['varname'] == 'tasmeanpoly':
                ds['degree'] = [f'tassumpoly_{k}' for k in ds.degree.values]
                ds = ds.rename({'degree':'pred'})
            elif params_proc[proc]['varname'] == 'tasmaxbinF':
                ds['binF'] = [f'tasmaxbinF_{k}' for k in ds.binF.values]
                ds = ds.rename({'binF':'pred'})
            elif params_proc[proc]['varname'] == 'days90plus':
                ds = ds.rename({'binF':'pred'})
            else:
                raise KeyError()

            # Rename proj_bases to allow for cross-processing
            #ds = ds.rename({'proj_base':'proj_base_bcds'})
        
            # Calculate impact from linear regression model
            ds[proc] = (coeffs.coeffs*ds[params_proc[proc]['varname']]).sum('pred',skipna=False)
            # Calculate change in impact vs. GWL0.61
            ds['d'+proc] = ds[proc] - ds[proc].sel(gwl=0.61)

            # Add max gwl to coordinates, for referece
            ds = ds.assign_coords({'maxgwl':(('idv'),ds.gwl.isel(gwl=np.array([(g-1) if g != 0 else -1 
                                                              for g in (np.isnan(ds['d'+proc]).
                                                                        argmax('gwl').
                                                                        max([d for d in ds['d'+proc].sizes if d not in ['gwl','idv']])).
                                                                        load()])).values)})
            # Make sure all-nan idvs are all-nan in maxgwl
            ds['maxgwl'] = ds.maxgwl.where(~np.isnan(ds['d'+proc]).all([d for d in ds['d'+proc].sizes if d != 'idv']).load())
        
            # Prepare output 
            ds_out = ds[['locid','d'+proc]]
            ds_out.attrs = {'SOURCE':'project_impacts.ipynb',
                    'DESCRIPTION':f'changes in {proc}, at the {params_proc[proc]['geo_typ']} level, projecting models fit on each of four reanalyses using each of four reanalysis-based bias-corrected and downscaled climate states',
                    'pop_weighted':params_proc[proc]['popwght']}
        
            # Export
            with warnings.catch_warnings():
                # Catches a Zarr v3 error related to string coordinates saved 
                # as objects instead of strings, I believe
                warnings.filterwarnings('ignore')
                utility_save(ds_out,output_fn,zarr_mode='w',save_kwargs = {'zarr_format':3,'consolidated':False})
        else:
            print(output_fn+' exists, skipped!')

  0%|          | 0/20 [00:00<?, ?it/s]

/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_ACCESS-CM2_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_ACCESS-CM2_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_MRI-ESM2-0_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_NESM3_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_NESM3_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!


/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_NorESM2-MM_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr removed to allow overwrite!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/impact_projs/payroll/dpayroll-crossfit_agg_NorESM2-MM_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr saved!
